# 📝 과제 01: 사내 IT 헬프데스크 팀 만들기

## 만들 결과

가상 회사의 **사내 IT 헬프데스크 팀**을 만듭니다. 슈퍼바이저가 계정·규정·조치·답변 담당 워커 중 필요한 워커를 차례로 골라 티켓을 처리합니다. 같은 팀이 티켓 내용에 따라 서로 다른 경로와 조치로 일하는지 확인하세요.

- 접수된 IT 티켓 3건(잠금 해제 요청 2건, 비밀번호 규정 문의 1건)을 처리합니다.
- 워커마다 필요한 도구만 받고, 계정을 바꾸거나 보안팀에 넘기는 조치는 한 워커만 할 수 있습니다.
- 티켓별 방문 순서, 실행된 조치 기록, 직원 답변을 `assignment01_runs.json`에 저장합니다.

교안 01(슈퍼바이저와 워커)의 구조를 적용합니다. `Command(goto, update)`, 워커별 도구 권한, 처리 상한이 핵심입니다.

<img src="images/assignment01_overview.png" width="1050" alt="슈퍼바이저가 계정·규정·조치·답변 담당 워커를 고르고, 워커는 보고를 들고 슈퍼바이저로 돌아오며, 슈퍼바이저가 끝낼 때 END로 갑니다.">

## 준비: 데이터와 제공 코드

- **자료:** `data/assignment01_*.json` 4개. 직원 계정 7개(`kim.minsu`, `park.jiho` 등)와 계정별 로그인 기록, 보안 규정 7개(`SEC-01`~`SEC-07`), 오늘 접수된 티켓 3건입니다(수업용 가상 자료). 티켓에는 요청자·채널·우선순위·접수 시각·본인 확인(사번+MFA) 여부·본문이 있습니다.
- **순서:** 분류 연습 -> 워커 에이전트 -> 슈퍼바이저 노드 -> 워커 노드 -> 그래프 연결 -> 실제 실행 -> 서술
- **제공:** 환경 설정, 도구, 워커 지시문, State, 슈퍼바이저 체인(`supervisor_chain`)과 `format_results`, `run_worker`, 나머지 워커 노드, `make_input`
- **작성:** `[작성]` 셀의 `...` 자리에 번호 주석대로 코드를 넣고, 바로 아래 자가채점 셀을 실행하세요. 자가채점은 모델을 호출하지 않습니다. 6번만 모델을 실제로 호출합니다.

In [ ]:
# 환경 설정: 라이브러리, 경로, API 키, 답변 모델
import os
import json
import re
from datetime import datetime
from pathlib import Path
from dotenv import find_dotenv, load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from IPython.display import Image, Markdown

# material_dir는 data·output이 있는 단원 폴더입니다. 정답에서는 ".."을 사용합니다.
material_dir = Path(".")
data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(parents=True, exist_ok=True)
load_dotenv(find_dotenv(usecwd=True))
# 단원 .env에 없는 공통 키는 저장소 최상위 .env에서 읽습니다. 이미 읽은 키는 덮어쓰지 않습니다.
load_dotenv(material_dir.resolve().parent / ".env")
model_name = os.getenv("OPENAI_CHAT_MODEL", "gpt-5.6-luna")
llm = ChatOpenAI(model=model_name, use_responses_api=True)
print("답변 모델:", model_name)

In [ ]:
# 그래프 정의와 자가채점에 필요한 클래스와 함수를 불러옵니다.
from copy import deepcopy
from unittest.mock import Mock, patch
from typing import Literal
from typing_extensions import TypedDict
from langchain.agents import create_agent
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command

In [ ]:
# [보조 함수] JSON 읽기: data 폴더의 JSON 파일을 파이썬 객체로 읽습니다.
def read_json(name):
    """data 폴더의 JSON 파일을 읽어 파이썬 객체로 반환합니다."""
    return json.loads((data_dir / name).read_text(encoding="utf-8"))


# [보조 함수] JSON 저장: 결과를 output 폴더의 JSON 파일로 저장합니다.
def save_json(name, value):
    """결과를 output 폴더의 JSON 파일로 저장합니다."""
    (output_dir / name).write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")


# 계정 데이터: 계정별 사번·이름·부서·직급·상태·잠금 사유·MFA 등록·마지막 비밀번호 변경일·관리자 여부
ACCOUNTS = read_json("assignment01_accounts.json")
# 로그인 기록: 계정별 최근 로그인(시각·IP·국가·사내망 여부·결과·실패 사유·기기)
LOGIN_EVENTS = read_json("assignment01_login_events.json")
# 보안 규정: 조항 ID(SEC-01~07)·제목·내용
POLICIES = read_json("assignment01_policies.json")
# 티켓 데이터: 티켓 ID별 제목·요청자·채널·우선순위·접수 시각·본인 확인 여부·본문
TICKETS = read_json("assignment01_tickets.json")

In [ ]:
# 오늘 접수된 티켓 3건을 확인합니다.
for ticket_id in TICKETS:
    row = TICKETS[ticket_id]
    if row["본인 확인"]:
        verified = "완료"
    else:
        verified = "미완료"

    print(ticket_id, row["제목"], "|", row["채널"], row["우선순위"], "| 본인 확인(사번+MFA):", verified)
    print("   ", row["본문"])

## 1. 티켓 본문을 보고 갈 곳과 State에 쓸 값을 함께 정하세요

`Command`의 `goto`는 다음에 갈 노드, `update`는 State에 쓸 값입니다. 둘을 한 번에 돌려줍니다. 팀 그래프에는 넣지 않는 워밍업입니다.

| 제공 코드 | 입력과 반환값 |
|---|---|
| `IntakeState` | `ticket`(티켓 본문), `account_id`, `category`를 가진 `TypedDict`입니다. |
| `find_account_id(text)` | 본문에서 `kim.minsu` 같은 계정 ID 문자열을 찾아 반환합니다. 없으면 `''`입니다. |

**작성:** `classify_ticket(state)` 노드를 완성하세요.

**목적:** 계정 ID가 있는 문의는 계정 담당이, 없는 문의는 규정 담당이 먼저 받게 합니다.

**구현할 처리**
1. `find_account_id`로 `state["ticket"]`에서 계정 ID를 찾습니다.
2. ID가 있으면 `account_worker`와 `"계정 문의"`, 없으면 `policy_worker`와 `"규정 문의"`로 정합니다.
3. `Command`를 반환합니다. 입력 state는 바꾸지 않습니다.

**반환값: `Command`**

| 키 | 반환할 값 |
|---|---|
| `goto` | `"account_worker"` 또는 `"policy_worker"` |
| `update["account_id"]` | 찾은 계정 ID. 없으면 `''` |
| `update["category"]` | `"계정 문의"` 또는 `"규정 문의"` |

**반환 예시:** 본문에 `kim.minsu`가 든 경우입니다.

```python
Command(goto="account_worker", update={"account_id": "kim.minsu", "category": "계정 문의"})
```

In [ ]:
# [State] 티켓 분류 연습의 공유 데이터
class IntakeState(TypedDict):
    ticket: str  # 티켓 본문
    account_id: str  # classify_ticket: 본문에서 찾은 계정 ID. 없으면 빈 문자열
    category: str  # classify_ticket: 문의 종류("계정 문의" 또는 "규정 문의")


# [보조 함수] 계정 ID 찾기: 본문에서 'kim.minsu' 같은 계정 ID를 찾습니다. 없으면 빈 문자열입니다.
def find_account_id(text):
    """티켓 본문에서 계정 ID를 찾아 반환합니다. 없으면 빈 문자열입니다."""
    found = re.search(r"[a-z]+\.[a-z]+", text)
    if found:
        return found.group()
    return ""

In [ ]:
# [작성] 본문에서 계정 ID를 찾아 먼저 받을 워커와 State에 쓸 값을 정하세요.
# [노드] 티켓 분류
def classify_ticket(state: IntakeState) -> Command[Literal["account_worker", "policy_worker"]]:
    """계정 ID가 있으면 account_worker로, 없으면 policy_worker로 보냅니다."""
    # 1. find_account_id로 본문에서 계정 ID를 찾으세요.
    ...

    # 2. ID가 있으면 account_worker와 "계정 문의", 없으면 policy_worker와 "규정 문의"로 정하세요.
    ...

    # 3. goto와 update를 담은 Command를 반환하세요.
    ...

In [ ]:
# [자가채점] 노드가 Command로 갈 곳과 State에 쓸 값을 함께 반환하는지 확인합니다.
expected = {"T-101": ("account_worker", "kim.minsu", "계정 문의"),
            "T-102": ("account_worker", "park.jiho", "계정 문의"),
            "T-103": ("policy_worker", "", "규정 문의")}
for ticket_id in expected:
    state = {"ticket": TICKETS[ticket_id]["본문"], "account_id": "", "category": ""}
    before = deepcopy(state)
    step = classify_ticket(state)
    assert isinstance(step, Command), "Command(goto=..., update=...)를 반환하세요. `...`를 채웠는지도 확인하세요."
    assert step.goto == expected[ticket_id][0], f"{ticket_id}: 계정 ID가 있으면 account_worker, 없으면 policy_worker로 보내세요."
    assert step.update["account_id"] == expected[ticket_id][1], f"{ticket_id}: update의 account_id를 확인하세요. ID가 없으면 빈 문자열입니다."
    assert step.update["category"] == expected[ticket_id][2], f"{ticket_id}: update의 category는 계정 ID가 있으면 '계정 문의', 없으면 '규정 문의'입니다."
    assert state == before, "입력 state는 바꾸지 마세요."
print("분류 노드 확인 완료")

## 2. 워커마다 필요한 도구만 주세요

워커는 받은 도구만 호출할 수 있습니다. 계정을 바꾸거나 보안팀에 넘기는 도구는 한 워커에게만 줍니다.

| 제공 도구 | 입력과 반환값 |
|---|---|
| `get_account(user_id)` | 사번·부서·직급·상태·잠금 사유·MFA 등록 여부·마지막 비밀번호 변경일·관리자 여부를 담은 dict |
| `get_login_events(user_id, limit=8)` | 최근 로그인 기록 목록(최신순). 시각·IP·국가·사내망 여부·결과·실패 사유·기기 |
| `search_policy(keyword)` | 제목이나 내용에 `keyword`가 든 규정 조항 목록(조항·제목·내용) |
| `unlock_account(user_id, ticket_id, reason)` | 티켓 요청자와 계정이 다르거나 본인 확인이 미완료이면 거부 문장, 아니면 잠금 해제 후 `ACTION_LOG`에 기록 |
| `reset_password(user_id, ticket_id)` | 같은 검사를 통과하면 임시 비밀번호를 SMS로 발송(비밀번호는 반환하지 않음)하고 기록 |
| `create_security_ticket(user_id, ticket_id, summary)` | 보안팀 확인 티켓을 접수하고 기록. 계정은 바꾸지 않음 |
| `get_reply_template(kind)` | `"처리 완료"`, `"보안 확인 필요"`, `"안내"` 중 하나의 답변 양식 |
| `ACCOUNT_PROMPT`, `POLICY_PROMPT`, `ACTION_PROMPT`, `REPLY_PROMPT` | 워커별 지시문 문자열 |

**작성:** 워커 에이전트 4개를 만드세요.

**목적:** 워커마다 일에 필요한 도구만 묶어 `create_agent(llm, tools=[...], system_prompt=...)`로 에이전트를 만듭니다.

**구현할 처리**
1. `account_agent`: `get_account`, `get_login_events`와 `ACCOUNT_PROMPT`를 줍니다.
2. `policy_agent`: `search_policy`와 `POLICY_PROMPT`를 줍니다.
3. `action_agent`: `unlock_account`, `reset_password`, `create_security_ticket`과 `ACTION_PROMPT`를 줍니다. 이 세 도구는 다른 워커에게 주지 않습니다.
4. `reply_agent`: `get_reply_template`과 `REPLY_PROMPT`를 줍니다.

**반환값:** 아래 변수 4개에 `create_agent`가 만든 에이전트를 담습니다.

| 변수 | 받는 도구 |
|---|---|
| `account_agent` | `get_account`, `get_login_events` |
| `policy_agent` | `search_policy` |
| `action_agent` | `unlock_account`, `reset_password`, `create_security_ticket` |
| `reply_agent` | `get_reply_template` |

In [ ]:
# 계정 담당 도구: 계정 정보와 로그인 기록을 조회만 합니다.
@tool
def get_account(user_id: str) -> dict:
    """계정의 사번·부서·직급·상태·잠금 사유·MFA 등록 여부·마지막 비밀번호 변경일·관리자 여부를 조회합니다."""
    if user_id not in ACCOUNTS:
        return {"안내": "없는 계정입니다."}

    return ACCOUNTS[user_id]


@tool
def get_login_events(user_id: str, limit: int = 8) -> list[dict]:
    """계정의 최근 로그인 기록을 최신순으로 limit건 조회합니다. 시각·IP·국가·사내망 여부·결과·실패 사유·기기가 있습니다."""
    if user_id not in LOGIN_EVENTS:
        return []

    # 기록은 최신순으로 저장되어 있으므로 앞에서 limit건을 자릅니다.
    return LOGIN_EVENTS[user_id][:limit]

In [ ]:
# 규정 담당 도구: 규정 제목·내용에 낱말이 들어 있는지만 보는 검색 도구입니다.
@tool
def search_policy(keyword: str) -> list[dict]:
    """제목이나 내용에 keyword가 들어간 보안 규정 조항을 찾습니다. keyword는 '잠금'처럼 짧은 낱말 하나입니다."""
    found = []
    for policy in POLICIES:
        if keyword in policy["제목"] or keyword in policy["내용"]:
            found.append(policy)

    return found

In [ ]:
# 조치 담당 도구: 계정을 바꾸거나 보안팀에 넘깁니다. 실행된 조치는 감사 기록(ACTION_LOG)에 남습니다.
# 티켓을 실행할 때마다 비워, 그 티켓에서 실행된 조치만 남깁니다.
ACTION_LOG = []


# 잠금 해제 도구입니다. 도구 안에서도 티켓의 요청자와 본인 확인을 검사합니다.
@tool
def unlock_account(user_id: str, ticket_id: str, reason: str) -> str:
    """티켓의 요청자가 계정 주인이고 본인 확인이 완료된 경우에만 계정 잠금을 해제하고 감사 기록을 남깁니다."""
    if ticket_id not in TICKETS:
        return f"거부: 없는 티켓입니다({ticket_id})."

    # 에이전트가 값을 잘못 넣어도, 요청자와 계정이 다르거나 본인 확인이 안 됐으면 도구가 거부합니다.
    if TICKETS[ticket_id]["요청자"] != user_id:
        return f"거부: 티켓 {ticket_id}의 요청자({TICKETS[ticket_id]['요청자']})와 계정({user_id})이 다릅니다."
    if not TICKETS[ticket_id]["본인 확인"]:
        return f"거부: 티켓 {ticket_id}는 본인 확인(사번+MFA)이 완료되지 않아 잠금을 해제할 수 없습니다."

    # 통과하면 감사 기록을 남기고 완료 문장을 반환합니다.
    ACTION_LOG.append({
        "ticket_id": ticket_id,
        "action": "unlock",
        "user_id": user_id,
        "at": datetime.now().isoformat(timespec="seconds"),
        "reason": reason,
    })

    return f"잠금 해제 완료: {user_id} (티켓 {ticket_id})"


# 비밀번호 초기화 도구입니다. 임시 비밀번호는 반환하지 않고 SMS로 따로 보냅니다.
@tool
def reset_password(user_id: str, ticket_id: str) -> str:
    """티켓의 요청자가 계정 주인이고 본인 확인이 완료된 경우에만 임시 비밀번호를 등록된 휴대폰으로 SMS 발송하고 감사 기록을 남깁니다."""
    if ticket_id not in TICKETS:
        return f"거부: 없는 티켓입니다({ticket_id})."

    if TICKETS[ticket_id]["요청자"] != user_id:
        return f"거부: 티켓 {ticket_id}의 요청자({TICKETS[ticket_id]['요청자']})와 계정({user_id})이 다릅니다."

    if not TICKETS[ticket_id]["본인 확인"]:
        return f"거부: 티켓 {ticket_id}는 본인 확인(사번+MFA)이 완료되지 않아 비밀번호를 초기화할 수 없습니다."

    # 임시 비밀번호는 반환하지 않고 기록만 남깁니다.
    ACTION_LOG.append({
        "ticket_id": ticket_id,
        "action": "reset_password",
        "user_id": user_id,
        "at": datetime.now().isoformat(timespec="seconds"),
    })

    return f"{user_id}의 임시 비밀번호를 등록된 휴대폰으로 SMS 발송했습니다."


# 보안팀에 확인을 요청하는 티켓을 만드는 도구입니다. 계정은 바꾸지 않습니다.
@tool
def create_security_ticket(user_id: str, ticket_id: str, summary: str) -> str:
    """비정상 로그인 같은 의심 상황을 보안팀 티켓으로 접수하고 감사 기록을 남깁니다."""
    ACTION_LOG.append({
        "ticket_id": ticket_id,
        "action": "security_ticket",
        "user_id": user_id,
        "at": datetime.now().isoformat(timespec="seconds"),
        "summary": summary,
    })

    return f"보안팀 확인 티켓 접수 완료: SR-{ticket_id} ({user_id})"

In [ ]:
# 답변 담당 도구: 답변 종류에 맞는 양식만 돌려줍니다. 답변에 쓸 사실은 전달받은 보고에서 가져옵니다.
@tool
def get_reply_template(kind: str) -> str:
    """직원 답변의 양식을 반환합니다. kind는 '처리 완료', '보안 확인 필요', '안내' 중 하나입니다."""
    if kind == "처리 완료":
        return """1. 문의 확인 인사 한 줄
2. 확인한 내용(계정 상태, 본인 확인, 로그인 기록)
3. 처리한 조치와 티켓 ID
4. 직원이 할 일(예: 다시 로그인)"""

    if kind == "보안 확인 필요":
        return """1. 문의 확인 인사 한 줄
2. 확인한 내용(본인 확인 상태, 비정상 로그인 기록)
3. 잠금을 해제하지 않은 이유(규정 조항 ID)와 보안팀 확인 요청을 접수한 사실
4. 직원이 할 일(보안팀 내선 7000 문의)과 보안팀 확인 뒤 해제된다는 안내"""

    return """1. 문의 확인 인사 한 줄
2. 질문에 해당하는 규정 조항 ID와 내용
3. 추가로 알아 둘 점"""

In [ ]:
# 워커별 지시문입니다.
ACCOUNT_PROMPT = (
    "사내 IT 헬프데스크의 계정 담당입니다. 지시받은 계정을 get_account와 get_login_events로 조회하고, "
    "계정 상태·MFA 등록·관리자 여부·최근 로그인 기록의 이상한 점(해외·외부 IP, 등록되지 않은 기기, 반복 실패 등)을 있는 그대로 짧게 보고하세요.")
POLICY_PROMPT = (
    "사내 IT 헬프데스크의 규정 담당입니다. search_policy는 낱말이 들어 있는지만 보므로 '잠금', '비정상', '변경'처럼 "
    "짧은 낱말 하나로 검색하세요. 찾은 규정의 조항 ID와 내용을 근거로 짧게 보고하고, 못 찾으면 다른 낱말로 다시 검색하세요.")
ACTION_PROMPT = (
    "사내 IT 헬프데스크의 조치 담당입니다. 지시받은 조치만 도구로 실행하세요. 도구에는 티켓 ID와 계정 ID를 넣고, "
    "도구가 거부하면 거부 사유를 그대로 보고하세요. 지시에 없는 조치는 하지 마세요. 결과를 짧게 보고하세요.")
REPLY_PROMPT = (
    "사내 IT 헬프데스크의 답변 작성 담당입니다. 조치를 완료했으면 '처리 완료', 보안팀 확인으로 넘겼으면 '보안 확인 필요', "
    "규정 질문이면 '안내'를 get_reply_template에 넣어 양식을 확인하세요. 전달받은 보고에 있는 사실만으로 "
    "직원에게 보낼 한국어 답변을 쓰고, 하지 않은 조치를 했다고 쓰지 마세요. 시간·기한 같은 숫자는 보고에 적힌 값만 쓰세요.")

In [ ]:
# [작성] 워커마다 필요한 도구만 묶어 에이전트를 만드세요.
# 1. account_agent: get_account, get_login_events와 ACCOUNT_PROMPT로 만드세요.
...

# 2. policy_agent: search_policy와 POLICY_PROMPT로 만드세요.
...

# 3. action_agent: 조치 도구 세 개(unlock_account, reset_password, create_security_ticket)와 ACTION_PROMPT로 만드세요.
...

# 4. reply_agent: get_reply_template과 REPLY_PROMPT로 만드세요.
...

In [ ]:
# [자가채점] 워커마다 받은 도구 이름을 확인합니다.
def tool_names(label):
    """변수 이름으로 에이전트를 찾아 가진 도구 이름을 정렬해 반환합니다."""
    assert label in globals(), f"{label}에 create_agent로 만든 에이전트를 담으세요."
    agent = globals()[label]
    assert hasattr(agent, "nodes"), f"{label}에 create_agent로 만든 에이전트를 담으세요."
    if "tools" not in agent.nodes:
        return []
    names = list(agent.nodes["tools"].bound.tools_by_name)
    names.sort()
    return names


assert tool_names("account_agent") == ["get_account", "get_login_events"], "account_agent는 조회 도구 get_account, get_login_events만 받습니다."
assert tool_names("policy_agent") == ["search_policy"], "policy_agent는 search_policy만 받습니다."
assert tool_names("action_agent") == ["create_security_ticket", "reset_password", "unlock_account"], "action_agent는 조치 도구 unlock_account, reset_password, create_security_ticket만 받습니다."
assert tool_names("reply_agent") == ["get_reply_template"], "reply_agent는 get_reply_template만 받습니다."
print("워커 도구 권한 확인 완료")

## 3. 슈퍼바이저가 다음 워커를 고르게 하세요

보고를 읽고 다음 워커와 지시를 한 번의 모델 호출로 정하고, 처리 상한을 지킵니다.

<img src="images/node_focus/assignment01_supervisor.png" width="1050" alt="팀 그래프에서 supervisor 노드만 강조합니다.">

| 제공 코드 | 입력과 반환값 |
|---|---|
| `TeamState` | `ticket`, `max_turns`, `instruction`, `visits`, `turn`, `results`를 가진 `TypedDict`입니다. |
| `supervisor_chain` | `{"ticket": ..., "results": ...}`를 받아 `Route` 객체를 반환합니다. `route.next`는 워커 이름 또는 `"FINISH"`, `route.instruction`은 지시입니다. |
| `format_results(results)` | 워커 보고 목록을 '[워커] 보고' 문단으로 묶습니다. |
| `CAP_INSTRUCTION` | 상한에 닿았을 때 `reply_writer`에게 줄 지시 문자열입니다. |

**작성:** `supervisor(state)` 노드를 완성하세요.

**목적:** 처리 상한을 지키며, 지금까지 보고를 읽고 다음 워커와 지시를 정합니다.

**구현할 처리**
1. `turn`이 `max_turns` 이상이면 모델을 호출하지 않습니다. `visits`에 `reply_writer`가 있으면 `END`로 끝내고, 없으면 `CAP_INSTRUCTION`을 주고 `reply_writer`로 보냅니다.
2. 상한 전이면 `supervisor_chain.invoke`를 한 번 호출합니다. 입력은 `state["ticket"]`과 `format_results(state["results"])`입니다.
3. `route.next`가 `"FINISH"`이면 `END`로 끝냅니다.
4. 아니면 `route.next`로 보내면서 지시와 방문 기록, 턴 수를 갱신합니다. 기존 목록을 직접 바꾸지 말고 새 목록을 만듭니다.

**반환값: `Command`**

| 키 | 반환할 값 |
|---|---|
| `goto` | 고른 워커 이름. 끝낼 때는 `END`, 상한에 닿았을 때는 `"reply_writer"` 또는 `END` |
| `update["instruction"]` | 다음 워커에게 줄 지시: `route.instruction`, 상한이면 `CAP_INSTRUCTION` |
| `update["visits"]` | 기존 방문 목록 뒤에 고른 워커(상한이면 `"reply_writer"`)를 더한 새 목록 |
| `update["turn"]` | `state["turn"] + 1` |

`END`로 끝낼 때는 `update` 없이 `Command(goto=END)`만 반환합니다.

**반환 예시:** `route.next`가 `"action_worker"`이고, 기존 `visits`가 `["account_worker"]`, `turn`이 1인 경우입니다.

```python
Command(goto="action_worker", update={
    "instruction": "티켓 T-101, 계정 kim.minsu의 잠금을 해제하세요.",
    "visits": ["account_worker", "action_worker"], "turn": 2
})
```

In [ ]:
# [State] 헬프데스크 그래프의 공유 데이터
class TeamState(TypedDict):
    # 처음 설정: 노드에서 바꾸지 않는 입력·한도
    ticket: str  # 티켓 정보(티켓 ID·요청자·채널·우선순위·접수 시각·본인 확인 여부·본문)
    max_turns: int  # 처리 상한: 워커에게 일을 맡길 수 있는 횟수. 닿으면 reply_writer에게 담당자 확인 안내를 한 번 더 맡깁니다

    # supervisor가 쓰는 필드: 다음 워커에게 줄 지시와 방문 기록
    instruction: str  # 이번 워커에게 준 지시
    visits: list[str]  # 일을 맡긴 워커 순서
    turn: int  # 일을 맡긴 횟수

    # 워커가 쓰는 필드: 리듀서가 없어 자동으로 덧붙지 않고, 반환한 목록으로 통째로 바뀝니다.
    # 그래서 워커가 지금까지의 보고에 새 보고를 붙인 목록 전체를 돌려줍니다(run_worker).
    results: list[dict]  # [{"worker": 워커 이름, "result": 보고}, ...]

In [ ]:
# 슈퍼바이저가 쓰는 팀 소개, 처리 원칙, 구조화 출력 체인입니다.
WORKER_GUIDE = (
    "account_worker: 계정 상태·MFA 등록·최근 로그인 기록 조회. "
    "policy_worker: 보안 규정 검색. "
    "action_worker: 잠금 해제(unlock_account), 비밀번호 초기화(reset_password), 보안팀 티켓 접수(create_security_ticket). "
    "reply_writer: 확인된 내용으로 직원에게 보낼 답변 작성."
)

# 팀의 처리 원칙: 워커를 부르는 순서는 정하지 않고 지켜야 할 기준만 적습니다.
# 누구를 언제 부를지는 슈퍼바이저가 지금까지의 보고를 읽고 정합니다.
TEAM_RULES = (
    "처리 원칙: "
    "1) action_worker의 조치(잠금 해제·비밀번호 초기화·보안팀 티켓)는 판단에 필요한 사실"
    "(티켓의 본인 확인 여부, 계정 상태와 최근 로그인 기록, 적용되는 보안 규정과 그 규정이 요구하는 조건)이 "
    "보고로 모두 확인된 뒤에만 맡깁니다. "
    "2) 판단에 필요한 사실이 아직 보고에 없으면, 그 사실을 알아낼 수 있는 워커에게 맡깁니다. 이미 보고된 내용은 다시 시키지 않습니다. "
    "3) 규정이 요구하는 조건을 채우지 못하면 계정을 바꾸지 말고, 규정이 정한 다른 처리를 따르고 "
    "reply_writer에게 직원이 할 일을 안내하게 합니다. "
    "4) 직원에게 보낼 답변이 나오면 FINISH를 고릅니다."
)

# 처리 상한에 닿았을 때 reply_writer에게 주는 지시입니다.
CAP_INSTRUCTION = (
    "처리 상한에 닿았습니다. 보고에 없는 조치를 했다고 쓰지 말고, 지금까지 확인한 내용과 함께 "
    "'헬프데스크 담당자가 이어서 확인한다'고 안내하세요."
)


# next는 그래프의 노드 이름과 같아야 그대로 goto에 쓸 수 있습니다.
class Route(BaseModel):
    next: Literal["account_worker", "policy_worker", "action_worker", "reply_writer", "FINISH"] = Field(
        description="다음에 일을 맡길 워커. reply_writer가 답변을 썼으면 FINISH")
    instruction: str = Field(description="다음 워커에게 줄 지시. 워커가 알아야 할 티켓 ID, 계정 ID, 앞 보고(상태·조항)를 넣는다. FINISH이면 빈 문자열")


supervisor_prompt = ChatPromptTemplate.from_messages([
    ("system", "사내 IT 헬프데스크 팀의 팀장입니다. 직접 조회하거나 처리하지 않고 워커에게 맡깁니다. "
     + WORKER_GUIDE + " " + TEAM_RULES +
     " 티켓과 지금까지 보고를 읽고 다음 워커 한 명과 지시를 정하세요."),
    ("human", "티켓:\n{ticket}\n\n지금까지 보고:\n{results}"),
])
supervisor_chain = supervisor_prompt | llm.with_structured_output(Route, strict=True)


# [보조 함수] LLM 컨텍스트 만들기: 워커 보고 목록을 supervisor 프롬프트와 워커에게 보내는 메시지에 넣을 '[워커] 보고' 문단으로 바꿉니다.
def format_results(results):
    """워커 보고 목록을 '[워커] 보고' 문단으로 묶습니다."""
    # 아직 보고가 없으면 첫 판단이라는 뜻으로 '아직 없음'을 넘깁니다.
    if not results:
        return "아직 없음"

    lines = []
    for row in results:
        lines.append(f"[{row['worker']}] {row['result']}")

    return "\n\n".join(lines)

In [ ]:
# [작성] 상한을 먼저 확인하고, 아니면 다음 워커를 골라 Command로 보내세요.
# [노드] 슈퍼바이저
def supervisor(state: TeamState) -> Command[Literal["account_worker", "policy_worker",
                                                   "action_worker", "reply_writer", END]]:
    """처리 상한을 지키며 다음 워커와 지시를 정해 Command로 보냅니다."""
    # 1. turn이 max_turns 이상이면 모델 없이 마무리하세요. visits에 reply_writer가 없으면 reply_writer로, 있으면 END로 보냅니다.
    ...

    # 2. 상한 전이면 supervisor_chain을 한 번 호출해 route를 받으세요.
    ...

    # 3. route.next가 "FINISH"이면 END로 끝내세요.
    ...

    # 4. 아니면 고른 워커로 보내면서 instruction·visits·turn을 갱신하세요.
    ...

In [ ]:
# [자가채점] 고정 응답으로 상한과 다음 워커 선택을 확인합니다.
def arg_of(call, index, name):
    """모의 호출 기록에서 위치 인자 또는 키워드 인자로 넘긴 값을 꺼냅니다."""
    if len(call.args) > index:
        return call.args[index]
    return call.kwargs[name]


def run_supervisor(state, route):
    """supervisor_chain을 고정 응답 route로 바꿔 supervisor를 한 번 실행합니다."""
    fake = Mock()
    fake.invoke.return_value = route
    with patch.dict(supervisor.__globals__, {"supervisor_chain": fake}):
        step = supervisor(state)
    return step, fake


base = {
    "ticket": "티켓 ID: T-900\n본문: kim.minsu 계정 잠금 해제 요청",
    "max_turns": 2,
    "instruction": "",
    "visits": [],
    "turn": 0,
    "results": [],
}
# 1) 첫 판단: 고른 워커로 가고, 지시·방문 기록·턴 수가 갱신됩니다.
state = deepcopy(base)
step, fake = run_supervisor(state, Route(next="account_worker", instruction="kim.minsu 계정을 조회하세요."))
assert isinstance(step, Command), "Command(goto=..., update=...)를 반환하세요. `...`를 채웠는지도 확인하세요."
assert fake.invoke.call_count == 1, "상한 전에는 supervisor_chain.invoke를 한 번 호출하세요."
assert arg_of(fake.invoke.call_args, 0, "input")["ticket"] == base["ticket"], "supervisor_chain.invoke에 ticket 키로 state['ticket']을 넘기세요."
assert step.goto == "account_worker", "route.next를 goto에 쓰세요."
assert step.update["instruction"] == "kim.minsu 계정을 조회하세요.", "update의 instruction에 route.instruction을 담으세요."
assert step.update["visits"] == ["account_worker"] and step.update["turn"] == 1, "visits에 고른 워커를 더하고 turn을 1 올리세요."
assert state == base, "입력 state의 목록을 직접 바꾸지 말고 새 목록을 만드세요."
# 2) 앞 보고가 있으면 chain 입력에 들어가고, 기록은 이어서 쌓입니다.
state = deepcopy(base)
state.update({"visits": ["account_worker"], "turn": 1,
              "results": [{"worker": "account_worker", "result": "상태 잠김, 사내망 로그인만 있음"}]})
step, fake = run_supervisor(state, Route(next="action_worker", instruction="잠금을 해제하세요."))
assert "상태 잠김, 사내망 로그인만 있음" in arg_of(fake.invoke.call_args, 0, "input")["results"], "format_results(state['results'])를 results 키로 넘기세요."
assert step.update["visits"] == ["account_worker", "action_worker"] and step.update["turn"] == 2, "기존 visits 뒤에 고른 워커를 더하고 turn을 1 올리세요."
# 3) FINISH이면 끝냅니다.
step, fake = run_supervisor(deepcopy(base), Route(next="FINISH", instruction=""))
assert step.goto == END, "FINISH이면 goto=END로 끝내세요."
# 4) 상한에 닿았고 답변 전이면 모델을 부르지 않고 reply_writer로 마무리합니다.
state = deepcopy(base)
state.update({"visits": ["account_worker", "policy_worker"], "turn": 2})
step, fake = run_supervisor(state, Route(next="account_worker", instruction=""))
assert fake.invoke.call_count == 0, "상한에 닿으면 supervisor_chain을 호출하지 마세요."
assert step.goto == "reply_writer", "상한에 닿고 답변 전이면 reply_writer로 보내세요."
assert step.update["instruction"] == CAP_INSTRUCTION, "상한용 지시 CAP_INSTRUCTION을 instruction에 담으세요."
assert step.update["visits"][-1] == "reply_writer" and step.update["turn"] == 3, "상한 마무리에서도 visits 끝에 'reply_writer'를 더하고 turn을 1 올리세요."
# 5) 상한에 닿았고 답변까지 끝났으면 바로 끝냅니다.
state = deepcopy(base)
state.update({"visits": ["account_worker", "reply_writer"], "turn": 2})
step, fake = run_supervisor(state, Route(next="account_worker", instruction=""))
assert fake.invoke.call_count == 0 and step.goto == END, "상한에 닿았고 답변을 이미 썼다면 END로 끝내세요."
print("슈퍼바이저 확인 완료")

## 4. 워커가 일하고 슈퍼바이저로 돌아가게 하세요

워커는 일을 마치면 보고를 `results`에 덧붙이고 `Command`로 슈퍼바이저에게 돌아갑니다.

<img src="images/node_focus/assignment01_account_worker.png" width="1050" alt="팀 그래프에서 account_worker 노드만 강조합니다.">

| 제공 코드 | 입력과 반환값 |
|---|---|
| `run_worker(agent, worker, state)` | 티켓·지시·앞 보고를 에이전트에 전달해 실행하고, 새 보고를 덧붙인 `results` 목록을 반환합니다. 입력 `state`는 바꾸지 않습니다. |

`results` 필드에는 리듀서가 없어 반환한 목록이 필드를 통째로 바꿉니다. 그래서 `run_worker`는 기존 보고에 새 보고를 붙인 목록 전체를 돌려줍니다. 슈퍼바이저가 한 번에 워커 한 명만 보내므로 같은 필드에 동시에 쓰는 일이 없습니다.

**작성:** `account_worker(state)` 노드를 완성하세요. 나머지 워커 노드는 5번에서 제공합니다.

**목적:** 계정 담당 에이전트를 실행하고 보고를 남긴 뒤 슈퍼바이저로 돌아갑니다.

**구현할 처리**
1. `run_worker(account_agent, "account_worker", state)`를 호출하고 반환값을 `results` 변수에 담습니다.
2. `Command`를 반환합니다. 입력 state는 바꾸지 않습니다.

**반환값: `Command`**

| 키 | 반환할 값 |
|---|---|
| `goto` | `"supervisor"` |
| `update["results"]` | `run_worker`가 반환한 `results` |

**반환 예시:** `run_worker`가 보고 한 건을 더한 목록을 반환한 경우입니다.

```python
Command(goto="supervisor", update={
    "results": [{"worker": "account_worker", "result": "상태 잠김, 사내망 로그인만 있음"}]
})
```

In [ ]:
# [보조 함수] 워커 실행: 지시와 지금까지의 보고로 워커 에이전트의 LLM 컨텍스트를 만들고, 실행한 뒤 기존 보고에 새 보고를 붙인 새 목록을 반환합니다.
def run_worker(agent, worker, state):
    """티켓·지시·지금까지 보고를 워커 에이전트에 전달하고, 새 보고를 덧붙인 results를 반환합니다."""
    # LLM 컨텍스트: 티켓, 이번 지시, 팀이 지금까지 확인한 내용. 워커끼리는 직접 대화하지 않으므로 앞 보고를 여기에 넣어 줍니다.
    message = (f"티켓:\n{state['ticket']}\n\n지시: {state['instruction']}\n\n"
               f"팀이 지금까지 확인한 내용:\n{format_results(state['results'])}")

    reply = agent.invoke({"messages": [{"role": "user", "content": message}]})

    # 마지막 메시지(워커의 최종 보고)만 꺼내 기존 보고 뒤에 붙인 새 목록을 반환합니다.
    return state["results"] + [{"worker": worker, "result": reply["messages"][-1].text}]

In [ ]:
# [작성] 계정 담당 에이전트를 실행하고 슈퍼바이저로 돌아가세요.
# [노드] 계정 담당 워커
def account_worker(state: TeamState) -> Command[Literal["supervisor"]]:
    """account_agent의 보고를 results에 더하고 supervisor로 돌아갑니다."""
    # 1. run_worker로 account_agent를 실행하고 새 results를 변수에 담으세요.
    ...

    # 2. supervisor로 돌아가는 Command를 반환하세요.
    ...

In [ ]:
# [자가채점] run_worker를 고정 응답으로 바꿔 워커 노드의 반환을 확인합니다.
def arg_of(call, index, name):
    """모의 호출 기록에서 위치 인자 또는 키워드 인자로 넘긴 값을 꺼냅니다."""
    if len(call.args) > index:
        return call.args[index]
    return call.kwargs[name]


def run_account_worker(state, new_results):
    """run_worker를 고정 응답으로 바꿔 account_worker를 한 번 실행합니다."""
    fake_run = Mock(return_value=new_results)
    with patch.dict(account_worker.__globals__, {"run_worker": fake_run}):
        step = account_worker(state)
    return step, fake_run


state = {"ticket": "티켓 ID: T-900", "max_turns": 8, "instruction": "kim.minsu를 조회하세요.",
         "visits": ["account_worker"], "turn": 1, "results": []}
before = deepcopy(state)
new_results = [{"worker": "account_worker", "result": "채점용 보고"}]
step, fake_run = run_account_worker(state, new_results)
assert isinstance(step, Command), "Command(goto=..., update=...)를 반환하세요. `...`를 채웠는지도 확인하세요."
assert fake_run.call_count == 1, "run_worker를 한 번 호출하세요."
assert arg_of(fake_run.call_args, 0, "agent") is account_agent, "첫 인자는 account_agent입니다."
assert arg_of(fake_run.call_args, 1, "worker") == "account_worker", "둘째 인자는 워커 이름 'account_worker'입니다."
assert step.goto == "supervisor", "워커는 일을 마치면 supervisor로 돌아갑니다."
assert step.update == {"results": new_results}, "run_worker가 반환한 results를 update의 results에 담으세요."
assert state == before, "입력 state는 바꾸지 마세요."
print("워커 노드 확인 완료")

## 5. 팀 그래프를 연결하세요

노드를 한 그래프에 등록합니다. 이동은 `Command`가 정하므로 시작 엣지만 등록합니다.

| 제공 코드 | 설명 |
|---|---|
| `policy_worker`, `action_worker`, `reply_writer` | 4번의 `account_worker`와 같은 모양의 워커 노드입니다. |

**작성:** 팀 그래프를 연결하고 이미지로 확인하세요.

**목적:** 슈퍼바이저와 워커 노드를 한 그래프로 묶고, 컴파일한 연결을 확인합니다.

**구현할 처리**
1. `TeamState`를 쓰는 `team_builder`를 만들고 노드 다섯 개를 등록합니다. 이름은 `supervisor`, `account_worker`, `policy_worker`, `action_worker`, `reply_writer`입니다.
2. `add_edge`는 `START`에서 `supervisor`로 가는 하나만 등록합니다.
3. `team_builder.compile()`의 결과를 `team_app`에 담습니다.
4. `get_graph().draw_mermaid_png()`와 `Image`, `display`로 그래프를 표시합니다.

**반환값:** 컴파일한 그래프를 `team_app` 변수에 담습니다. 이미지에서 슈퍼바이저와 워커 사이에 점선이 보여야 합니다. 점선이 없으면 3번과 4번 노드의 반환 타입 힌트를 확인하세요.

In [ ]:
# 나머지 워커 노드입니다. account_worker와 같은 모양입니다.
# [노드] 규정 담당 워커
def policy_worker(state: TeamState) -> Command[Literal["supervisor"]]:
    """policy_agent의 보고를 results에 더하고 supervisor로 돌아갑니다."""
    results = run_worker(policy_agent, "policy_worker", state)
    return Command(goto="supervisor", update={"results": results})


# [노드] 조치 담당 워커
def action_worker(state: TeamState) -> Command[Literal["supervisor"]]:
    """action_agent의 보고를 results에 더하고 supervisor로 돌아갑니다."""
    results = run_worker(action_agent, "action_worker", state)
    return Command(goto="supervisor", update={"results": results})


# [노드] 답변 작성 워커
def reply_writer(state: TeamState) -> Command[Literal["supervisor"]]:
    """reply_agent의 보고를 results에 더하고 supervisor로 돌아갑니다."""
    results = run_worker(reply_agent, "reply_writer", state)
    return Command(goto="supervisor", update={"results": results})

In [ ]:
# [작성] 노드를 등록하고 시작 엣지만 연결한 뒤 이미지로 확인하세요.
# [그래프 조립] 슈퍼바이저와 워커 넷을 노드로 등록하고 시작 엣지만 연결합니다.
# 1. TeamState로 team_builder를 만들고 노드 다섯 개를 등록하세요.
...

# 2. START에서 supervisor로 가는 시작 엣지만 등록하세요.
...

# 3. compile한 결과를 team_app에 담으세요.
...

# 4. 컴파일한 그래프를 이미지로 표시하세요.
...

In [ ]:
# [자가채점] 컴파일한 그래프의 노드와 연결을 확인합니다.
assert "team_app" in globals(), "compile한 결과를 team_app 변수에 담으세요."
graph = team_app.get_graph()
workers = ["account_worker", "policy_worker", "action_worker", "reply_writer"]
for name in ["supervisor"] + workers:
    assert name in graph.nodes, f"{name} 노드를 add_node로 등록하세요."
pairs = []
for edge in graph.edges:
    pairs.append((edge.source, edge.target, edge.conditional))
    if edge.source != "__start__":
        assert edge.conditional, f"{edge.source} -> {edge.target}는 add_edge로 등록하지 마세요. 이동은 Command가 정합니다."
assert ("__start__", "supervisor", False) in pairs, "START에서 supervisor로 가는 시작 엣지를 등록하세요."
for worker in workers:
    assert ("supervisor", worker, True) in pairs, f"supervisor에서 {worker}로 가는 연결이 없습니다. supervisor 노드 반환 타입 힌트의 Literal을 확인하세요."
    assert (worker, "supervisor", True) in pairs, f"{worker}에서 supervisor로 돌아오는 연결이 없습니다. 워커 노드 반환 타입 힌트 Command[Literal['supervisor']]를 확인하세요."
assert ("supervisor", "__end__", True) in pairs, "supervisor에서 END로 가는 연결이 없습니다. supervisor 노드 반환 타입 힌트에 END를 넣으세요."
print("그래프 연결 확인 완료")

## 6. 티켓 3건을 실제로 실행하고 저장하세요

같은 팀이 티켓마다 다른 경로와 조치로 일하는지 실제 모델로 확인하고 처리 기록을 남깁니다. 이 문항에서만 모델을 호출합니다.

| 제공 코드 | 입력과 반환값 |
|---|---|
| `TICKETS` | 티켓 ID를 키로 하는 dict입니다. |
| `make_input(ticket_id)` | 티켓 ID를 받아 팀의 처음 State dict를 반환합니다. |
| `ACTION_LOG` | 실행된 조치의 감사 기록 목록입니다. 항목은 `ticket_id`, `action`(`unlock`·`reset_password`·`security_ticket`), `user_id`, `at`을 가집니다. |
| `save_json(name, value)` | 값을 `output` 폴더의 JSON 파일로 저장합니다. |

**작성:** 세 티켓을 실행하고 `runs`에 결과를 담아 저장하세요.

**목적:** 티켓마다 방문 순서·조치 기록·직원 답변을 모아 처리 결과를 남깁니다.

**구현할 처리**
1. 결과를 담을 `runs` dict를 만듭니다.
2. 티켓마다 `ACTION_LOG`를 비우고, `team_app.invoke(make_input(ticket_id))`를 한 번 호출합니다.
3. `runs[ticket_id]`에 방문 순서, 조치 기록의 복사본, 마지막 보고를 담습니다.
4. `save_json("assignment01_runs.json", runs)`로 저장합니다.

**반환값: `dict`** `runs[ticket_id]`는 아래 세 키를 가진 dict입니다.

| 키 | 담을 값 |
|---|---|
| `visits` | 일을 맡긴 워커 순서 목록(`result["visits"]`) |
| `actions` | 그 티켓에서 실행된 조치 기록의 복사본(`list(ACTION_LOG)`) |
| `reply` | 마지막 워커의 보고, 직원에게 보낼 답변(`result["results"][-1]["result"]`) |

**반환 예시:** T-103을 실행한 경우입니다.

```python
{"T-103": {"visits": ["policy_worker", "reply_writer"], "actions": [], "reply": "안녕하세요. ..."}}
```

**확인 기준:** 자가채점이 아래를 확인합니다.
- 세 티켓 모두 마지막 방문이 `reply_writer`이고 방문은 최대 9회(처리 상한 8회와 상한 뒤 답변 1회)입니다. 조치 기록의 `ticket_id`는 그 티켓의 ID입니다.
- T-101: `account_worker`가 `action_worker`보다 먼저(규정을 확인했다면 `policy_worker`도 먼저)이고, 조치는 `unlock` 한 건(계정 kim.minsu)입니다.
- T-102: `account_worker`와 `policy_worker`가 `action_worker`보다 먼저이고, `unlock` 없이 `security_ticket` 한 건(계정 park.jiho)이며 답변에 '보안팀'이 있습니다.
- T-103: `policy_worker`를 거치고 조치가 없습니다.
- 저장한 `assignment01_runs.json`이 `runs`와 같습니다.

모델 응답을 고정하거나 결과를 고쳐 쓰지 마세요. 문장과 경로는 모델 판단에 따라 달라지므로, 코드가 맞는데 자가채점이 실패하면 실행 셀부터 다시 실행해 보세요.

In [ ]:
# [보조 함수] 그래프 시작 입력 만들기: 티켓 한 건으로 모든 필드의 초깃값을 채웁니다. 보고·방문 기록은 비우고 턴은 0부터 셉니다.
def make_input(ticket_id, max_turns=8):
    """티켓 ID를 받아 티켓 정보 문자열과 빈 기록을 담은 처음 State를 반환합니다."""
    row = TICKETS[ticket_id]
    # 본인 확인 여부를 사람이 읽는 문구로 바꿉니다.
    if row["본인 확인"]:
        verified = "완료"
    else:
        verified = "미완료"
    # 슈퍼바이저와 워커가 읽을 티켓 정보 문자열입니다.
    ticket = (f"티켓 ID: {ticket_id}\n요청자: {row['요청자']}\n채널: {row['채널']}\n우선순위: {row['우선순위']}\n"
              f"접수 시각: {row['접수 시각']}\n본인 확인(사번+MFA): {verified}\n본문: {row['본문']}")

    # State의 모든 필드를 채워 반환합니다.
    return {
        "ticket": ticket,
        "max_turns": max_turns,
        "instruction": "",
        "visits": [],
        "turn": 0,
        "results": [],
    }

In [ ]:
# [작성] 세 티켓을 실제로 실행하고 결과를 저장하세요.
# 1. 결과를 담을 runs dict를 만드세요.
runs = {}
for ticket_id in TICKETS:
    # 2. ACTION_LOG를 비우고 make_input(ticket_id)로 team_app을 한 번 실행하세요.
    ...

    # 3. runs[ticket_id]에 visits·actions·reply를 담으세요.
    ...

# 4. runs를 assignment01_runs.json으로 저장하세요.
...

In [ ]:
# [자가채점] 저장된 결과로 방문 순서와 조치 기록을 확인합니다. 모델은 호출하지 않습니다.
assert "runs" in globals() and runs != {}, "runs에 티켓별 결과를 담으세요."
RETRY = "코드가 맞다면 모델 판단이 달라진 것일 수 있습니다. supervisor 노드가 format_results(state['results'])를 chain에 넘기는지 확인하고 6번 실행 셀과 이 셀을 다시 실행해 보세요."
kinds_of = {}
for ticket_id in TICKETS:
    assert ticket_id in runs, f"runs에 {ticket_id} 결과가 없습니다. 티켓 3건을 모두 실행하세요."
    run = runs[ticket_id]
    assert run["visits"] and run["visits"][-1] == "reply_writer", f"{ticket_id}: 마지막 워커는 reply_writer여야 합니다. {RETRY}"
    # 처리 상한 8회 안에서 일하고, 상한에 닿으면 reply_writer를 1번 더 부르므로 최대 9회입니다.
    assert len(run["visits"]) <= 9, f"{ticket_id}: 방문은 처리 상한 8회와 상한 뒤 답변 1회, 최대 9회 안에서 끝나야 합니다."
    assert run["reply"].strip(), f"{ticket_id}: 직원 답변(마지막 보고)을 reply에 담으세요."
    kinds_of[ticket_id] = []
    for row in run["actions"]:
        assert row["ticket_id"] == ticket_id, f"{ticket_id}: 다른 티켓의 조치가 섞였습니다. 티켓마다 ACTION_LOG.clear()로 비우세요."
        kinds_of[ticket_id].append(row["action"])
assert runs["T-101"]["actions"] is not runs["T-103"]["actions"], "actions에는 list(ACTION_LOG)로 복사본을 담으세요."
first, second, third = runs["T-101"], runs["T-102"], runs["T-103"]
# 확인이 조치보다 먼저입니다: 계정 확인은 조치 전에, 규정 확인을 했다면 그것도 조치 전에 있어야 합니다.
for ticket_id in ["T-101", "T-102"]:
    visits = runs[ticket_id]["visits"]
    assert "account_worker" in visits and "action_worker" in visits, f"{ticket_id}: account_worker와 action_worker를 모두 거쳐야 합니다. {RETRY}"
    assert visits.index("account_worker") < visits.index("action_worker"), f"{ticket_id}: 계정 확인(account_worker)이 조치(action_worker)보다 먼저여야 합니다. {RETRY}"
    if "policy_worker" in visits:
        assert visits.index("policy_worker") < visits.index("action_worker"), f"{ticket_id}: 규정 확인(policy_worker)이 조치(action_worker)보다 먼저여야 합니다. {RETRY}"
# T-101: 본인 확인이 끝난 티켓은 잠금을 해제합니다.
assert kinds_of["T-101"] == ["unlock"], f"T-101은 unlock 조치가 한 건만 기록되어야 합니다. {RETRY}"
assert first["actions"][0]["user_id"] == "kim.minsu", "T-101의 조치 대상 계정은 kim.minsu여야 합니다."
# T-102: 해외 로그인 의심 티켓은 규정을 확인하고 잠금 해제 대신 보안 티켓을 접수합니다.
assert "policy_worker" in second["visits"], f"T-102는 policy_worker(규정 확인)를 거쳐야 합니다. {RETRY}"
assert "unlock" not in kinds_of["T-102"], "T-102는 잠금을 해제하면 안 됩니다. 도구의 거부 검사와 슈퍼바이저 지시를 확인하세요."
assert kinds_of["T-102"] == ["security_ticket"], f"T-102는 security_ticket 조치가 한 건만 기록되어야 합니다. {RETRY}"
assert second["actions"][0]["user_id"] == "park.jiho", "T-102의 조치 대상 계정은 park.jiho여야 합니다."
assert "보안팀" in second["reply"], f"T-102 답변에는 보안팀 확인 안내가 있어야 합니다. {RETRY}"
# T-103: 규정 문의는 규정을 확인하고 조치 없이 답변합니다.
assert "policy_worker" in third["visits"], f"T-103은 policy_worker를 거쳐야 합니다. {RETRY}"
assert third["actions"] == [], f"T-103은 조치를 실행하면 안 됩니다. {RETRY}"
saved = json.loads((output_dir / "assignment01_runs.json").read_text(encoding="utf-8"))
assert saved["T-101"]["visits"] == first["visits"] and saved["T-103"]["reply"] == third["reply"], "save_json으로 runs를 assignment01_runs.json에 저장하세요."
print("티켓 3건 실행 결과 확인 완료")

T-102에 대한 직원 답변입니다. 잠금을 해제했다고 쓰지 않고, 보안팀 확인 요청을 접수한 사실과 보안팀 내선 7000 문의를 안내하는지 읽어 보세요.

In [ ]:
# T-102에 대한 직원 답변을 읽어 봅니다.
display(Markdown(runs["T-102"]["reply"]))

## 7. 경로와 안전장치가 막아 준 것을 설명하세요

**작성:** 서술형 답안

**목적:** 슈퍼바이저의 판단, 도구 권한, 도구 안의 검사, 처리 상한이 각각 어떤 위험을 줄이는지 설명합니다.

**설명할 내용**
1. T-102에서 슈퍼바이저는 어떤 보고를 읽고 `unlock_account` 대신 `create_security_ticket`을 맡겼습니까?
2. 슈퍼바이저가 잘못 판단해 `unlock_account`를 요청해도 T-102의 잠금이 풀리지 않는 이유를 쓰세요. 조치 도구를 `action_worker`에게만 준 것은 무엇을 막습니까?
3. `max_turns` 상한은 무엇을 막습니까? 상한에 닿았을 때 확정 답변 대신 `CAP_INSTRUCTION` 안내로 마무리하는 이유는 무엇입니까?

정답 노트북의 모범 설명과 비교하세요.

*(여기에 확인한 근거와 판단 이유를 작성하세요.)*